# 04 · Knowledge Graph Design & Construction
Goal: store each customer's six months of behaviour as a **temporal graph** in Neo4j.

```
(:Customer)-[:HAS_ACCOUNT]->(:CreditAccount)-[:HAS_MONTHLY_RECORD]->(:MonthlyCreditRecord)
(:MonthlyCreditRecord)-[:NEXT_MONTH {coverage, util_delta, status_change}]->(:MonthlyCreditRecord)
(:MonthlyCreditRecord)-[:IN_STATE]->(:RepaymentState)
```
* **Payment coverage lives on the `NEXT_MONTH` edge**, because a payment in month t+1 settles the bill of month t. It describes the step between two months, not either month alone.
* **5 shared `RepaymentState` nodes** connect all 180,000 monthly records. They let us look at the whole population's month-to-month movements (notebook 05).
* **The default label is never put into the graph**, so no Cypher query can "cheat" by reading it.
* No demographic nodes: age/sex/education/marriage stay as plain properties for description only.

**Setup.** Credentials come from the `.env` file (`NEO4J_URI`, `NEO4J_USERNAME`, `NEO4J_PASSWORD`) through the official Neo4j Python driver.

In [1]:
import sys, pathlib, warnings
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
from credit_kg import config as C
import time
from credit_kg import data as D, graph as G

**Connect and check the server.**

In [2]:
driver = G.get_driver()
G.run(driver, "CALL dbms.components() YIELD name, versions, edition RETURN name, versions, edition")

,name,versions,edition
0,Neo4j Kernel,[5.26.32],community


**Safety check.** Stop if the database contains labels from another project, so we never mix graphs.

In [3]:
other = G.foreign_labels(driver)
assert not other, f"Database contains other labels {other}; use an empty database for this project."
print("database is clean or contains only this project's labels")

database is clean or contains only this project's labels


**Prepare the rows to load** from the notebook-01 tables: 30,000 customers, 180,000 monthly records and 150,000 month-to-month edges (5 per customer).

In [4]:
wide, monthly, split = D.load_processed()
rows = G.build_rows(wide, monthly, split)
{k: len(v) for k, v in rows.items()}

{'states': 5, 'customers': 30000, 'records': 180000, 'next_month': 150000}

**Look at one edge before loading.** Customer 1, August → September: the September payment (0) against the August bill (3,102) gives coverage 0. The customer stays in the 2+ months late state.

In [5]:
[e for e in rows["next_month"] if e["from_id"] == "1-5"]

[{'from_id': '1-5',
  'to_id': '1-6',
  'coverage': 0.0,
  'util_delta': 0.04055,
  'transition': 'LATE_2PLUS->LATE_2PLUS',
  'status_change': 'stable'}]

**Load the graph.** First we create uniqueness constraints (these stop duplicates and make lookups fast). Then everything is added with `MERGE` in batches of 5,000 rows, so running the loader again updates nodes instead of duplicating them.

In [6]:
t0 = time.time()
timings = G.load_graph(driver, rows)
print("seconds per pass:", timings, "| total", round(time.time() - t0, 1))
counts_first = G.graph_counts(driver)
counts_first

seconds per pass: {'constraints': 0.48935651779174805, 'states': 0.6, 'customers': 6.5, 'records': 23.0, 'next_month': 10.5} | total 41.1


{'customers': 30000,
 'accounts': 30000,
 'monthly_records': 180000,
 'states': 5,
 'HAS_ACCOUNT': 30000,
 'HAS_MONTHLY_RECORD': 180000,
 'NEXT_MONTH': 150000,
 'IN_STATE': 180000}

**Integrity check 1: exact counts.** Expected: 30,000 customers and accounts, 180,000 monthly records, 5 states, 150,000 `NEXT_MONTH` edges, and 180,000 `IN_STATE` links.

In [7]:
expected = {"customers": 30000, "accounts": 30000, "monthly_records": 180000, "states": 5,
            "HAS_ACCOUNT": 30000, "HAS_MONTHLY_RECORD": 180000, "NEXT_MONTH": 150000, "IN_STATE": 180000}
assert counts_first == expected, counts_first
print("all counts match")

all counts match


**Integrity check 2: temporal order.** Every account has exactly 6 months, and every `NEXT_MONTH` edge goes from month t to month t+1 of the *same* customer.

In [8]:
bad_accounts = G.run(driver, '''
MATCH (a:CreditAccount)-[:HAS_MONTHLY_RECORD]->(m)
WITH a, count(m) AS months, collect(DISTINCT m.month_index) AS idx
WHERE months <> 6 OR size(idx) <> 6 RETURN count(a) AS n''')["n"][0]
bad_edges = G.run(driver, '''
MATCH (x:MonthlyCreditRecord)-[:NEXT_MONTH]->(y:MonthlyCreditRecord)
WHERE y.month_index <> x.month_index + 1 OR y.customer_id <> x.customer_id RETURN count(*) AS n''')["n"][0]
print("accounts without exactly 6 months:", bad_accounts, "| out-of-order edges:", bad_edges)
assert bad_accounts == 0 and bad_edges == 0

accounts without exactly 6 months: 0 | out-of-order edges: 0


**Integrity check 3: no target in the graph.** We list every property actually stored on our nodes and relationships. None may hold the default label.

In [9]:
node_keys = G.run(driver, "MATCH (n) UNWIND keys(n) AS k RETURN DISTINCT k")["k"]
rel_keys = G.run(driver, "MATCH ()-[r]->() UNWIND keys(r) AS k RETURN DISTINCT k")["k"]
keys = set(node_keys) | set(rel_keys)
leaky = {k for k in keys if "default" in k.lower() or k.lower() in {"target", "label", "y"}}
print("properties stored in the graph:", sorted(keys))
assert not leaky, leaky

properties stored in the graph: ['account_id', 'age', 'bill_amount', 'code', 'coverage', 'credit_balance', 'credit_limit', 'customer_id', 'description', 'education_code', 'is_late', 'marriage_code', 'month', 'month_index', 'ordinal', 'payment_amount', 'record_id', 'repayment_status', 'sex_code', 'split', 'state', 'status_change', 'transition', 'util_delta', 'utilization']


**Integrity check 4: values match the source table.** We pull 200 random monthly records back out of Neo4j and compare them with the pandas table.

In [10]:
sample = monthly.sample(200, random_state=C.SEED)
back = G.run(driver, '''UNWIND $ids AS id MATCH (m:MonthlyCreditRecord {record_id: id})
RETURN m.record_id AS record_id, m.bill_amount AS bill_amount, m.repayment_status AS repayment_status, m.utilization AS utilization''',
             ids=sample.record_id.tolist())
cmp = sample.merge(back, on="record_id", suffixes=("", "_graph"))
assert len(cmp) == 200 and np.allclose(cmp.bill_amount, cmp.bill_amount_graph) and (cmp.repayment_status == cmp.repayment_status_graph).all() \
    and np.allclose(cmp.utilization, cmp.utilization_graph)
print("200/200 sampled records match")

200/200 sampled records match


**Idempotency test.** We run the whole loader a second time. Because of `MERGE` and the constraints, all counts must stay exactly the same (no duplicates).

In [11]:
G.load_graph(driver, rows)
counts_second = G.graph_counts(driver)
assert counts_second == counts_first, (counts_first, counts_second)
print("second load changed nothing:", counts_second == counts_first)

second load changed nothing: True


**One customer's six-month chain as stored in the graph** (customer 1: two months late at the end, nothing paid against the August bill).

In [12]:
G.run(driver, '''
MATCH (c:Customer {customer_id: 1})-[:HAS_ACCOUNT]->(a)-[:HAS_MONTHLY_RECORD]->(m)-[:IN_STATE]->(s)
OPTIONAL MATCH (m)-[n:NEXT_MONTH]->(nxt)
RETURN m.month AS month, a.credit_limit AS limit, m.bill_amount AS bill, m.payment_amount AS paid,
       s.code AS state, round(m.utilization, 3) AS util,
       n.coverage AS coverage_of_this_bill_next_month, n.status_change AS next_change
ORDER BY m.month_index''')

,month,limit,bill,paid,state,util,coverage_of_this_bill_next_month,next_change
0,2005-04,20000.0,0.0,0.0,NO_USE,0.000,NaN,stable
1,2005-05,20000.0,0.0,0.0,NO_USE,0.000,NaN,stable
2,2005-06,20000.0,0.0,0.0,PAID_FULL,0.000,NaN,stable
3,2005-07,20000.0,689.0,0.0,PAID_FULL,0.034,1.0,worsening
4,2005-08,20000.0,3102.0,689.0,LATE_2PLUS,0.155,0.0,stable
5,2005-09,20000.0,3913.0,0.0,LATE_2PLUS,0.196,NaN,NaN


**Try it yourself in Neo4j Browser** (http://localhost:7474), for example:
```
MATCH p=(:Customer {customer_id: 1})-[:HAS_ACCOUNT]->()-[:HAS_MONTHLY_RECORD]->(:MonthlyCreditRecord {month_index: 1})-[:NEXT_MONTH*5]->()
RETURN p
```

**Close the connection.**

In [13]:
driver.close()

### Takeaways
* 240,005 nodes and 540,000 relationships, loaded in about 20 seconds in batches, with constraints created first.
* The loader is **idempotent**: running it twice gives identical counts.
* Temporal order is enforced and checked (`month_index`, `NEXT_MONTH` only t → t+1).
* The graph holds behaviour only. The default label stays outside, so the patterns found in notebook 05 are real evidence, not a lookup.